In [3]:
import numpy as np
from numpy import array
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
import seaborn as sns
import pandas as pd
import tensorflow as tf

# Seed for reproducibility
np.random.seed(42)

Dataset: https://www.kaggle.com/datasets/behrad3d/nasa-cmaps?resource=download

Strategy: Training data constains multivariate time series, where each engine is run until a fault. Therefore the target ouput in the training set is the difference between the last cycle and the timesteps.

In [4]:
dataset_path = "datasets"

# Number of sensors in the dataset
NUM_SENSORS = 21

# Column names for the dataframe
columns = ["engine_id", "cycle", "op_1", "op_2", "op_3", *[f"sensor_{i}" for i in range(1,NUM_SENSORS+1)]]

# Creating dataframes
# rul_df is the ground truth of the test_df:
#   - rul[0] is the RUL for the last timestep for engine_0
train_df = pd.read_csv(f"{dataset_path}/Train/train_FD001.txt",sep=r"\s+", header=None, names=columns)
test_df = pd.read_csv(f"{dataset_path}/Test/test_FD001.txt", sep=r"\s+", header=None, names=columns)
rul_df = pd.read_csv(f"{dataset_path}/RUL_FD001.txt", sep=r"\s+", header=None, names=["RUL"])

# Size of the datasets.
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"RUL shape: {rul_df.shape}")


Train shape: (20631, 26)
Test shape: (13096, 26)
RUL shape: (100, 1)


By looking at the head and tail of the dataframe, we can extract the following:
1. There are 100 engines run to fault.
2. We need to prune variables, as there might be varaibles that are constant, i.e they did not contribute to the fault or degradation in RUL.
3. There variables that are magnitudes larger than others:
    - We need to normalize the column values to bring them to a comparable scale.

In [5]:
# Displaying the first 5 rows
train_df.head()

,engine_id,cycle,op_1,op_2,op_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [6]:
# Displaying the last 5 rows
train_df.tail()

,engine_id,cycle,op_1,op_2,op_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21
20626,100,196,-0.0004,-0.0003,100.0,518.67,643.49,1597.98,1428.63,14.62,...,519.49,2388.26,8137.60,8.4956,0.03,397,2388,100.0,38.49,22.9735
20627,100,197,-0.0016,-0.0005,100.0,518.67,643.54,1604.50,1433.58,14.62,...,519.68,2388.22,8136.50,8.5139,0.03,395,2388,100.0,38.30,23.1594
20628,100,198,0.0004,0.0000,100.0,518.67,643.42,1602.46,1428.18,14.62,...,520.01,2388.24,8141.05,8.5646,0.03,398,2388,100.0,38.44,22.9333
20629,100,199,-0.0011,0.0003,100.0,518.67,643.23,1605.26,1426.53,14.62,...,519.67,2388.23,8139.29,8.5389,0.03,395,2388,100.0,38.29,23.0640
20630,100,200,-0.0032,-0.0005,100.0,518.67,643.85,1600.38,1432.14,14.62,...,519.30,2388.26,8137.33,8.5036,0.03,396,2388,100.0,38.37,23.0522


In [ ]:
# Checking for NAN values in the dataset
print("NAN in Train:", train_df.isnull().sum().sum())
print("NAN in Test:", test_df.isnull().sum().sum())

NAN in Train: 0
NAN in Test: 0


In [16]:
# Checking for constant, or near constant, variables.
# The following columns can be removed:
#   - op_1-op_3, sensor_1, sensor_10,sensor_18 and sensor_19

train_df.describe().T

,count,mean,std,min,25%,50%,75%,max
engine_id,20631.0,51.506568,2.922763e+01,1.0000,26.0000,52.0000,77.0000,100.0000
cycle,20631.0,108.807862,6.888099e+01,1.0000,52.0000,104.0000,156.0000,362.0000
op_1,20631.0,-0.000009,2.187313e-03,-0.0087,-0.0015,0.0000,0.0015,0.0087
op_2,20631.0,0.000002,2.930621e-04,-0.0006,-0.0002,0.0000,0.0003,0.0006
op_3,20631.0,100.000000,0.000000e+00,100.0000,100.0000,100.0000,100.0000,100.0000
sensor_1,20631.0,518.670000,0.000000e+00,518.6700,518.6700,518.6700,518.6700,518.6700
sensor_2,20631.0,642.680934,5.000533e-01,641.2100,642.3250,642.6400,643.0000,644.5300
sensor_3,20631.0,1590.523119,6.131150e+00,1571.0400,1586.2600,1590.1000,1594.3800,1616.9100
sensor_4,20631.0,1408.933782,9.000605e+00,1382.2500,1402.3600,1408.0400,1414.5550,1441.4900
sensor_5,20631.0,14.620000,1.776400e-15,14.6200,14.6200,14.6200,14.6200,14.6200


In [69]:
# Removing columns that are constant, or near constant
columns = ["op_1", "op_2", "op_3", "sensor_1", "sensor_10","sensor_18", "sensor_19"]

for column in columns:
    train_df = train_df.drop(columns = [f"{column}"])
    test_df = test_df.drop(columns = [f"{column}"])

train_df.head()

KeyError: "['op_1'] not found in axis"

Adding target variable RUL, for each engine, in the training data. It will be calculated using this formula: 

\begin{equation}
RUL = \text{last\_cycle - current\_cycle}
\end{equation}

In addition to this, RUL will saturated to 150 to avoid prediciting large RUL values during training. It is more critical to correctly predict RUL when we are closer to failure, for maintance reasons, than predicing when a maintance is in the far distance.

In [56]:
RUL = []
for row_index in range(len(train_df)):
    current_cycle = train_df.iloc[row_index]["cycle"] # Current cycle of the engine
    engine_id = train_df.iloc[row_index]["engine_id"] # Engine_id
    last_cycle = len(train_df[train_df["engine_id"] == engine_id]) # Extracts the last cycle value for the engine
    RUL.append(last_cycle-current_cycle) # RUL at each cycle

# Adding RUL to the training set
train_df.insert(2, "RUL", RUL)

ValueError: cannot insert RUL, already exists

Normalizing the variables, except engine_id, cycles, and RUL as they are not inputs to the system

In [62]:
columns_to_normalize = [column for column in train_df.columns if column not in ["engine_id", "cycle", "RUL"]]
scaler = StandardScaler()

train_df[columns_to_normalize] = scaler.fit_transform(train_df[columns_to_normalize] )
test_df[columns_to_normalize] = scaler.transform(test_df[columns_to_normalize] )

train_df.head()

,engine_id,cycle,RUL,sensor_2,sensor_3,sensor_4,sensor_5,sensor_6,sensor_7,sensor_8,sensor_9,sensor_11,sensor_12,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_20,sensor_21
0,1,1,191.0,-1.721725,-0.134255,-0.925936,0.0,0.141683,1.121141,-0.516338,-0.862813,-0.266467,0.334262,-1.058890,-0.269071,-0.603816,0.0,-0.781710,1.348493,1.194427
1,1,2,190.0,-1.061780,0.211528,-0.643726,0.0,0.141683,0.431930,-0.798093,-0.958818,-0.191583,1.174899,-0.363646,-0.642845,-0.275852,0.0,-0.781710,1.016528,1.236922
2,1,3,189.0,-0.661813,-0.413166,-0.525953,0.0,0.141683,1.008155,-0.234584,-0.557139,-1.015303,1.364721,-0.919841,-0.551629,-0.649144,0.0,-2.073094,0.739891,0.503423
3,1,4,188.0,-0.661813,-1.261314,-0.784831,0.0,0.141683,1.222827,0.188048,-0.713826,-1.539489,1.961302,-0.224597,-0.520176,-1.971665,0.0,-0.781710,0.352598,0.777792
4,1,5,187.0,-0.621816,-1.251528,-0.301518,0.0,0.141683,0.714393,-0.516338,-0.457059,-0.977861,1.052871,-0.780793,-0.521748,-0.339845,0.0,-0.136018,0.463253,1.059552


In the C-MAPSS dataset challenge, we are given the RUL for the last recorded cycle. Therefore the test data is the last row for each engine in the test_df, and the target is the RUL_df.

In [68]:
X_train = train_df[columns_to_normalize]
y_train = train_df["RUL"]

X_test = train_df.groupby("engine_id").last().reset_index() #Extracts the last row for each engine
y_test = rul_df["RUL"] # RUL value for the last cycle in the train_df


print(f"Training Data: X={X_train.shape}, y={y_train.shape}")
print(f"Testing Data:  X={X_test.shape},  y={y_test.shape}")

Training Data: X=(20631, 17), y=(20631,)
Testing Data:  X=(100, 20),  y=(100,)
